# use models to segment your images
a notebook to apply image segmentation on your images

in Colab, we need to connect to a GPU

on the top right, go to change runtime type and select T4 GPU

In [ ]:
pip install readlif

In [ ]:
pip install scikit-image

In [ ]:
pip install matplotlib

It is normal that in colab, you might neeed to restart the session when you attempt to pip install apoc.... Simply confirm restart of the session and start from the first executable cell

In [ ]:
!pip install apoc --no-deps
!pip install "scikit-learn" "pyclesperanto-prototype" "pandas"
#!pip install "numpy==2.4.4"

In [ ]:
from matplotlib import pyplot as plt
import apoc
from skimage.io import imread, imsave
import numpy as np
from readlif.reader import LifFile

In [ ]:
import os
# Clone the repo if not already in Colab
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('/content/NBCimageAnalysis'):
        !git clone https://github.com/FilLieb/NBCimageAnalysis.git
    os.chdir('/content/NBCimageAnalysis/learning/')
    print(os.listdir('.'))

We have 4 channels and assign their names to a list:

In [ ]:
channels = ["DAAO", "Cre", "vGAT", "Gphn"]  # change index to switch between channels

In [ ]:
# set paths
image_folder = '../data/2026group2/'

In [ ]:
def execute(file, root):
    print(f"Found image: {file}")
    folder_name = os.path.splitext(os.path.basename(file))[0] # removes the .lif extension
    folder_path = os.path.join(root, folder_name)
    os.makedirs(folder_path, exist_ok=True)

    lif = LifFile(file)

    for img in lif.get_iter_image():
        print(img.name, img.dims)
        channels_array = np.stack([np.array(channel) for channel in img.get_iter_c(t=0, z=0)])
        make_masks("DAAO", channels_array[0], folder_path, img.name)
        make_masks("vGAT", channels_array[1], folder_path, img.name)
        make_masks("Cre", channels_array[2], folder_path, img.name)
        make_masks("Gphn", channels_array[3], folder_path, img.name)


In [ ]:
# make mask per channel
def make_masks(channel, array, path, image):
    print("Generating masks...")

    out_file = os.path.join(path, image + '_' + channel + '_labels.tif')
    
    cl_filename = '../training_4_channels/' + channel + '/models/' + channel + '_object_model.cl'
    segmenter = apoc.ObjectSegmenter(opencl_filename=cl_filename)
    labels = segmenter.predict(array)

    imsave(out_file, labels)

In [ ]:
for root, dirs, files in os.walk(image_folder):
    for file in files:
        if file.endswith("High Density.lif"):
            full_path = os.path.join(root, file)
            print(full_path)
            execute(full_path, root)

print("...completed.")


In [ ]:
for root, dirs, files in os.walk(image_folder):
    for file in files:
        if file.endswith("_labels.tif"):
            label_path = os.path.join(root, file)
            label = imread(label_path)
            folder_name = os.path.basename(root)
            plt.figure()
            plt.imshow(label)
            plt.title("predicted labels - " + folder_name + ": " + file)
            plt.show()
